# Lending Message A/B Test: Part A and Part B

This notebook analyzes the lending message experiment from `data/applicants_experiment.csv` and keeps only Part A and Part B for a GitHub-ready submission.

## Part A - Experiment Analysis

Primary outcome: `doc_submitted_72h`, because the business goal is to get applicants to submit missing income documents. Clicks and responses are secondary diagnostics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, chisquare
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

app = pd.read_csv('data/applicants_experiment.csv')
variants = ['original', 'A', 'B']
primary = 'doc_submitted_72h'

print(app.info())
display(app.head())

In [ ]:
duplicate_applicants = app['applicant_id'].duplicated().sum()
print(f'Duplicate applicant IDs: {duplicate_applicants}')
display(app.groupby('variant').size().reindex(variants).rename('n'))

### Sample Ratio Mismatch Check

This checks whether the observed assignment counts are consistent with an equal split across variants.

In [ ]:
observed = app['variant'].value_counts().reindex(variants)
expected = np.repeat(len(app) / len(variants), len(variants))
srm_chi2, srm_p_value = chisquare(observed.values, expected)
srm_check = pd.DataFrame({
    'variant': variants,
    'observed_n': observed.values,
    'expected_n': expected,
    'observed_share': observed.values / len(app),
    'expected_share': expected / len(app),
    'difference_n': observed.values - expected,
    'chi2_stat': srm_chi2,
    'p_value': srm_p_value,
})
display(srm_check.style.format({
    'expected_n': '{:.0f}',
    'observed_share': '{:.2%}',
    'expected_share': '{:.2%}',
    'difference_n': '{:+.0f}',
    'chi2_stat': '{:.4f}',
    'p_value': '{:.4f}',
}))

### Group Balance Checks

In [ ]:
display(pd.crosstab(app['variant'], app['channel'], normalize='index').reindex(variants))
display(pd.crosstab(app['variant'], app['risk_band'], normalize='index').reindex(variants))
display(pd.crosstab(app['variant'], app['missing_doc_type'], normalize='index').reindex(variants))

In [ ]:
balance_rows = []
for variable in ['channel', 'risk_band', 'missing_doc_type']:
    table = pd.crosstab(app['variant'], app[variable]).reindex(variants)
    chi2_stat, p_value, dof, _ = chi2_contingency(table)
    shares = table.div(table.sum(axis=1), axis=0)
    balance_rows.append({
        'variable': variable,
        'chi2_stat': chi2_stat,
        'dof': dof,
        'p_value': p_value,
        'max_cell_share_spread': (shares.max(axis=0) - shares.min(axis=0)).max(),
    })
balance_summary = pd.DataFrame(balance_rows)
display(balance_summary.style.format({
    'chi2_stat': '{:.4f}',
    'p_value': '{:.4f}',
    'max_cell_share_spread': '{:.2%}',
}))

### Power Analysis

This checks whether the observed A vs Original effect was large enough to detect with the available sample. It reports power at alpha 0.05 and at the Bonferroni-adjusted alpha of 0.0167 used for the three primary pairwise comparisons.

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

control = app[app['variant'] == 'original'][primary]
treatment = app[app['variant'] == 'A'][primary]

n_control = control.count()
n_treat = treatment.count()
control_rate = control.mean()
treatment_rate = treatment.mean()
abs_lift = treatment_rate - control_rate
alpha_adjusted = 0.05 / 3
effect_size = abs(proportion_effectsize(treatment_rate, control_rate))
ratio = n_treat / n_control
power_model = NormalIndPower()

required_n_control = power_model.solve_power(
    effect_size=effect_size,
    power=0.80,
    alpha=alpha_adjusted,
    ratio=ratio,
    alternative='two-sided'
)
achieved_power_alpha_0_05 = power_model.power(
    effect_size=effect_size,
    nobs1=n_control,
    alpha=0.05,
    ratio=ratio,
    alternative='two-sided'
)
achieved_power_bonferroni_alpha = power_model.power(
    effect_size=effect_size,
    nobs1=n_control,
    alpha=alpha_adjusted,
    ratio=ratio,
    alternative='two-sided'
)

power_summary = pd.DataFrame([{
    'comparison': 'A vs original',
    'control_rate': control_rate,
    'treatment_rate': treatment_rate,
    'abs_lift': abs_lift,
    'effect_size_cohens_h': effect_size,
    'alpha_adjusted': alpha_adjusted,
    'n_control_observed': n_control,
    'n_treat_observed': n_treat,
    'achieved_power_alpha_0_05': achieved_power_alpha_0_05,
    'achieved_power_bonferroni_alpha': achieved_power_bonferroni_alpha,
    'required_n_control_for_80pct_power': required_n_control,
    'required_n_treat_for_80pct_power': required_n_control * ratio,
}])
display(power_summary.style.format('{:.4f}', subset=[
    'control_rate', 'treatment_rate', 'abs_lift', 'effect_size_cohens_h',
    'alpha_adjusted', 'achieved_power_alpha_0_05',
    'achieved_power_bonferroni_alpha', 'required_n_control_for_80pct_power',
    'required_n_treat_for_80pct_power'
]))

### Reusable Statistical Test

All pairwise statistical results below are calculated with `hypothesis_test`. It accepts explicit control and treatment dataframes, runs a two-sided two-proportion z-test, and returns the result as a one-row dataframe with conversion rates, lift, p-value, and a 95% confidence interval for the treatment-control difference.

In [ ]:
def hypothesis_test(
    control_variant,
    treatment_variant,
    metric,
    alpha=0.05,
    control_df=None,
    treatment_df=None,
):
    if control_df is None:
        control_df = app[app['variant'] == control_variant]
    if treatment_df is None:
        treatment_df = app[app['variant'] == treatment_variant]

    control = control_df[metric]
    treatment = treatment_df[metric]

    n_control = control.count()
    n_treat = treatment.count()
    x_control = control.sum()
    x_treat = treatment.sum()

    control_rate = x_control / n_control
    treatment_rate = x_treat / n_treat
    abs_lift = treatment_rate - control_rate
    rel_lift = abs_lift / control_rate if control_rate != 0 else np.nan

    z_stat, p_value = proportions_ztest(
        count=[x_treat, x_control],
        nobs=[n_treat, n_control],
        alternative='two-sided'
    )
    ci_low, ci_high = confint_proportions_2indep(
        count1=x_treat,
        nobs1=n_treat,
        count2=x_control,
        nobs2=n_control,
        method='wald',
        compare='diff',
        alpha=alpha
    )

    return pd.DataFrame([
        {
            'comparison': f'{treatment_variant} vs {control_variant}',
            'control_variant': control_variant,
            'treatment_variant': treatment_variant,
            'metric': metric,
            'n_control': n_control,
            'n_treat': n_treat,
            'x_control': x_control,
            'x_treat': x_treat,
            'control_rate': control_rate,
            'treatment_rate': treatment_rate,
            'abs_lift': abs_lift,
            'rel_lift': rel_lift,
            'z_stat': z_stat,
            'p_value': p_value,
            'ci_low': ci_low,
            'ci_high': ci_high,
        }
    ])

### Reusable Hypothesis Test Visualization

Use `plot_hypothesis_results` to visualize the output dataframe from `hypothesis_test`. The `significance_level` input controls the p-value reference line and the significant/non-significant color coding.

In [ ]:
def plot_hypothesis_results(
    test_results,
    significance_level=0.05,
    title='Hypothesis test results',
):
    results = test_results.copy()
    if isinstance(results, pd.Series):
        results = results.to_frame().T
    elif isinstance(results, dict):
        results = pd.DataFrame([results])

    required_cols = {'comparison', 'abs_lift', 'ci_low', 'ci_high', 'p_value'}
    missing_cols = required_cols - set(results.columns)
    if missing_cols:
        raise ValueError(f'Missing required columns: {sorted(missing_cols)}')

    results = results.reset_index(drop=True)
    if 'segment' in results.columns:
        labels = results['segment'].astype(str)
    elif 'metric' in results.columns and results['metric'].nunique() > 1:
        labels = results['metric'].astype(str) + ': ' + results['comparison'].astype(str)
    else:
        labels = results['comparison'].astype(str)

    results = results.assign(
        label=labels,
        significant=results['p_value'] < significance_level,
        abs_lift_pp=results['abs_lift'] * 100,
        ci_low_pp=results['ci_low'] * 100,
        ci_high_pp=results['ci_high'] * 100,
    ).iloc[::-1]

    y = np.arange(len(results))
    colors = np.where(results['significant'], '#2563eb', '#6b7280')
    fig_height = max(3.2, 0.55 * len(results) + 1.8)
    fig, (ax_ci, ax_p) = plt.subplots(
        1,
        2,
        figsize=(12, fig_height),
        gridspec_kw={'width_ratios': [1.45, 1]},
        sharey=True,
    )

    effects = results['abs_lift_pp'].to_numpy(dtype=float)
    ci_low = results['ci_low_pp'].to_numpy(dtype=float)
    ci_high = results['ci_high_pp'].to_numpy(dtype=float)
    xerr = np.vstack([effects - ci_low, ci_high - effects])

    ax_ci.errorbar(
        effects,
        y,
        xerr=xerr,
        fmt='none',
        ecolor='#374151',
        elinewidth=1.5,
        capsize=4,
        zorder=1,
    )
    ax_ci.scatter(effects, y, color=colors, s=70, zorder=2)
    ax_ci.axvline(0, color='#111827', linestyle='--', linewidth=1)
    ax_ci.set_yticks(y)
    ax_ci.set_yticklabels(results['label'])
    ax_ci.set_xlabel('Absolute lift, percentage points')
    ax_ci.set_title('Lift with confidence interval')
    ax_ci.grid(axis='x', alpha=0.25)

    p_values = results['p_value'].clip(lower=1e-12).to_numpy(dtype=float)
    min_x = max(min(p_values.min(), significance_level) / 5, 1e-6)
    max_x = min(max(p_values.max(), significance_level) * 2, 1)
    ax_p.hlines(y, min_x, p_values, color=colors, linewidth=2, alpha=0.65)
    ax_p.scatter(p_values, y, color=colors, s=70, zorder=2)
    ax_p.axvline(significance_level, color='#dc2626', linestyle='--', linewidth=1.2)
    ax_p.set_xscale('log')
    ax_p.set_xlim(min_x, max_x)
    ax_p.set_xlabel('p-value, log scale')
    ax_p.set_title(f'p-value vs alpha={significance_level:.4g}')
    ax_p.grid(axis='x', alpha=0.25)
    ax_p.tick_params(axis='y', labelleft=False)

    for row_y, p_value in zip(y, p_values):
        ax_p.annotate(
            f'{p_value:.3g}',
            xy=(p_value, row_y),
            xytext=(6, 0),
            textcoords='offset points',
            va='center',
            fontsize=9,
        )

    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()
    return fig, (ax_ci, ax_p)

### Primary Outcome

In [ ]:
primary_summary = app.groupby('variant')[primary].agg(successes='sum', n='count', rate='mean').reindex(variants)
control_rate = primary_summary.loc['original', 'rate']
primary_summary['abs_lift_vs_original'] = primary_summary['rate'] - control_rate
primary_summary['rel_lift_vs_original'] = primary_summary['abs_lift_vs_original'] / control_rate
display(primary_summary.style.format('{:.4f}', subset=['rate', 'abs_lift_vs_original', 'rel_lift_vs_original']))

### Multiple-Testing Policy

The three planned primary-outcome pairwise tests use Bonferroni correction. Guardrail and segment tests are diagnostic checks and should be interpreted directionally rather than as winner-selection tests.

In [ ]:
primary_tests = pd.concat([
    hypothesis_test('original', 'A', primary, control_df=app[app['variant'] == 'original'], treatment_df=app[app['variant'] == 'A']),
    hypothesis_test('original', 'B', primary, control_df=app[app['variant'] == 'original'], treatment_df=app[app['variant'] == 'B']),
    hypothesis_test('B', 'A', primary, control_df=app[app['variant'] == 'B'], treatment_df=app[app['variant'] == 'A']),
], ignore_index=True)
primary_tests['p_bonferroni'] = (primary_tests['p_value'] * 3).clip(upper=1)
primary_tests['significant_bonferroni'] = primary_tests['p_bonferroni'] < 0.05
display(primary_tests[['comparison','n_control','n_treat','x_control','x_treat','control_rate','treatment_rate','abs_lift','rel_lift','z_stat','p_value','p_bonferroni','significant_bonferroni','ci_low','ci_high']].style.format('{:.4f}', subset=['control_rate','treatment_rate','abs_lift','rel_lift','z_stat','p_value','p_bonferroni','ci_low','ci_high']))

### Visualization: Statistical Test Results

In [ ]:
plot_hypothesis_results(
    primary_tests,
    significance_level=0.05 / 3,
    title='Primary outcome pairwise tests',
)

### Delivered-Only Sensitivity Check

In [ ]:
delivered_app = app[app['delivered'] == 1].copy()
delivered_summary = delivered_app.groupby('variant')[primary].agg(successes='sum', n='count', rate='mean').reindex(variants)
delivered_control_rate = delivered_summary.loc['original', 'rate']
delivered_summary['abs_lift_vs_original'] = delivered_summary['rate'] - delivered_control_rate
delivered_summary['rel_lift_vs_original'] = delivered_summary['abs_lift_vs_original'] / delivered_control_rate
display(delivered_summary.style.format('{:.4f}', subset=['rate', 'abs_lift_vs_original', 'rel_lift_vs_original']))

delivered_tests = pd.concat([
    hypothesis_test('original', 'A', primary, control_df=delivered_app[delivered_app['variant'] == 'original'], treatment_df=delivered_app[delivered_app['variant'] == 'A']),
    hypothesis_test('original', 'B', primary, control_df=delivered_app[delivered_app['variant'] == 'original'], treatment_df=delivered_app[delivered_app['variant'] == 'B']),
    hypothesis_test('B', 'A', primary, control_df=delivered_app[delivered_app['variant'] == 'B'], treatment_df=delivered_app[delivered_app['variant'] == 'A']),
], ignore_index=True)
delivered_tests['p_bonferroni'] = (delivered_tests['p_value'] * 3).clip(upper=1)
delivered_tests['significant_bonferroni'] = delivered_tests['p_bonferroni'] < 0.05
display(delivered_tests[['comparison','n_control','n_treat','x_control','x_treat','control_rate','treatment_rate','abs_lift','rel_lift','z_stat','p_value','p_bonferroni','significant_bonferroni','ci_low','ci_high']].style.format('{:.4f}', subset=['control_rate','treatment_rate','abs_lift','rel_lift','z_stat','p_value','p_bonferroni','ci_low','ci_high']))

### Visualization: Primary Outcome

In [ ]:
primary_plot = primary_summary.copy()
primary_plot['ci_low'] = [
    max(0, rate - 1.96 * np.sqrt(rate * (1 - rate) / n))
    for rate, n in zip(primary_plot['rate'], primary_plot['n'])
]
primary_plot['ci_high'] = [
    min(1, rate + 1.96 * np.sqrt(rate * (1 - rate) / n))
    for rate, n in zip(primary_plot['rate'], primary_plot['n'])
]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#6b7280', '#2563eb', '#f97316']
labels = ['Original', 'Rewrite A', 'Rewrite B']
x = np.arange(len(variants))
rates = primary_plot['rate'].values * 100
yerr = np.vstack((
    (primary_plot['rate'] - primary_plot['ci_low']).values * 100,
    (primary_plot['ci_high'] - primary_plot['rate']).values * 100,
))
bars = ax.bar(x, rates, color=colors, width=0.62)
ax.errorbar(x, rates, yerr=yerr, fmt='none', ecolor='#111827', capsize=5)
ax.axhline(primary_plot.loc['original', 'rate'] * 100, color='#6b7280', linestyle='--')
ax.set_xticks(x, labels)
ax.set_ylabel('Document submitted within 72h (%)')
ax.set_title('Primary outcome by variant')
ax.set_ylim(0, max((primary_plot['ci_high'] * 100).max() + 3, 24))
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.7, f'{rate:.1f}%', ha='center', va='bottom')
plt.tight_layout()
plt.show()

### Guardrails

In [ ]:
guardrails = ['unsub_7d', 'complaint_7d', 'support_contact_7d']
guardrail_rates = app.groupby('variant')[guardrails].mean().reindex(variants)
guardrail_counts = app.groupby('variant')[guardrails].sum().reindex(variants)
display(guardrail_rates.style.format('{:.4%}'))
display(guardrail_counts)

guardrail_tests = pd.concat([
    hypothesis_test(
        control,
        treatment,
        metric,
        control_df=app[app['variant'] == control],
        treatment_df=app[app['variant'] == treatment],
    )
    for metric in guardrails
    for control, treatment in [('original', 'A'), ('original', 'B'), ('B', 'A')]
], ignore_index=True)
display(guardrail_tests[['metric','comparison','control_rate','treatment_rate','abs_lift','rel_lift','z_stat','p_value','ci_low','ci_high']].style.format('{:.4f}', subset=['control_rate','treatment_rate','abs_lift','rel_lift','z_stat','p_value','ci_low','ci_high']))

### Visualization: Guardrails

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
metric_labels = ['Unsubscribe', 'Complaint', 'Support contact']
width = 0.24
x = np.arange(len(guardrails))
colors = ['#6b7280', '#2563eb', '#f97316']
labels = ['Original', 'Rewrite A', 'Rewrite B']

for i, variant in enumerate(variants):
    values = guardrail_rates.loc[variant].values * 100
    ax.bar(x + (i - 1) * width, values, width=width, label=labels[i], color=colors[i])
    for j, value in enumerate(values):
        ax.text(j + (i - 1) * width, value + 0.08, f'{value:.2f}%', ha='center', fontsize=8)

ax.set_xticks(x, metric_labels)
ax.set_ylabel('7-day rate (%)')
ax.set_title('Guardrail rates by variant')
ax.legend(frameon=False)
ax.set_ylim(0, max(guardrail_rates.max().max() * 100 + 1.2, 4))
plt.tight_layout()
plt.show()

### Segment Review

In [ ]:
for col in ['channel', 'risk_band', 'missing_doc_type']:
    segment = app.groupby([col, 'variant'])[primary].agg(successes='sum', n='count', rate='mean').reset_index()
    control = segment[segment['variant'] == 'original'][[col, 'rate']].rename(columns={'rate':'original_rate'})
    segment = segment.merge(control, on=col)
    segment['abs_lift_vs_original'] = segment['rate'] - segment['original_rate']
    print('\n' + col)
    display(segment.pivot(index=col, columns='variant', values='rate').style.format('{:.2%}'))
    display(segment.pivot(index=col, columns='variant', values='abs_lift_vs_original').style.format('{:.2%}'))

### Visualization: Segment Lift

In [ ]:
segment_frames = []
for col in ['channel', 'risk_band', 'missing_doc_type']:
    segment = app.groupby([col, 'variant'])[primary].agg(successes='sum', n='count', rate='mean').reset_index()
    control = segment[segment['variant'] == 'original'][[col, 'rate']].rename(columns={'rate': 'original_rate'})
    segment = segment.merge(control, on=col)
    segment['abs_lift_vs_original'] = segment['rate'] - segment['original_rate']
    segment = segment[segment['variant'].isin(['A', 'B'])].copy()
    segment['segment'] = col + ': ' + segment[col].astype(str)
    segment_frames.append(segment[['segment', 'variant', 'abs_lift_vs_original']])

segment_plot = pd.concat(segment_frames, ignore_index=True)
segment_pivot = segment_plot.pivot(index='segment', columns='variant', values='abs_lift_vs_original').sort_index()

fig, ax = plt.subplots(figsize=(9, 5.8))
y = np.arange(len(segment_pivot.index))
ax.axvline(0, color='#111827', linewidth=1)
ax.barh(y - 0.18, segment_pivot['A'] * 100, height=0.32, label='Rewrite A', color='#2563eb')
ax.barh(y + 0.18, segment_pivot['B'] * 100, height=0.32, label='Rewrite B', color='#f97316')
ax.set_yticks(y, segment_pivot.index)
ax.set_xlabel('Absolute lift vs Original (percentage points)')
ax.set_title('Primary outcome lift by segment')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

### Segment Hypothesis Tests

In [ ]:
segment_tests = pd.concat([
    hypothesis_test(
        'original',
        'A',
        primary,
        control_df=app[(app['risk_band'] == 'high') & (app['variant'] == 'original')],
        treatment_df=app[(app['risk_band'] == 'high') & (app['variant'] == 'A')],
    ).assign(segment='High risk'),
    hypothesis_test(
        'original',
        'A',
        primary,
        control_df=app[(app['missing_doc_type'] == 'both') & (app['variant'] == 'original')],
        treatment_df=app[(app['missing_doc_type'] == 'both') & (app['variant'] == 'A')],
    ).assign(segment='Missing both docs'),
    hypothesis_test(
        'original',
        'A',
        primary,
        control_df=app[(app['channel'] == 'sms') & (app['variant'] == 'original')],
        treatment_df=app[(app['channel'] == 'sms') & (app['variant'] == 'A')],
    ).assign(segment='SMS channel'),
], ignore_index=True)

segment_tests = segment_tests[
    ['segment', 'comparison', 'n_control', 'n_treat', 'control_rate',
     'treatment_rate', 'abs_lift', 'rel_lift', 'p_value', 'ci_low', 'ci_high']
]

display(segment_tests.style.format('{:.3f}', subset=[
    'control_rate', 'treatment_rate', 'abs_lift', 'rel_lift',
    'p_value', 'ci_low', 'ci_high'
]).set_table_styles([
    {'selector': 'th', 'props': [('white-space', 'nowrap'), ('width', 'auto')]},
    {'selector': 'td', 'props': [('white-space', 'nowrap'), ('width', 'auto')]},
]))

### Visualization: Segment Hypothesis Tests

In [ ]:
plot_hypothesis_results(
    segment_tests,
    significance_level=0.05,
    title='Priority segment hypothesis tests',
)

## Part B - Written Recommendation

Recommend Variant A as the winning candidate. Variant A improves `doc_submitted_72h` from 17.18% to 19.36%, an absolute lift of 2.17 percentage points and a relative lift of 12.65% versus Original. The A vs Original confidence interval for the lift is positive, and the result remains statistically significant after Bonferroni correction for the three pairwise primary-outcome comparisons.

Variant B should not be rolled out. It has a smaller and statistically inconclusive primary-outcome lift, while its unsubscribe and support-contact rates are directionally worse than Original.

Recommendation: roll out Variant A with monitoring rather than rolling out B. Monitor unsubscribe, complaint, and support-contact guardrails after launch. Segment-level results should be treated as directional because each subgroup has less sample than the overall experiment.

## Drawing conclusions

Variant A is the strongest candidate because it improves the primary outcome meaningfully and remains acceptable on the reviewed guardrails.